In [ ]:
# Set required environment variables for Azure OpenAI
import os
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://ducksat.cognitiveservices.azure.com/openai/deployments/gpt-5-nano/chat/completions?api-version=2025-01-01-preview"
os.environ["AZURE_OPENAI_API_KEY"] = "CKKtDWZjp3uujnxsx9mO4D2pafqAgxXnMy7vCSdaDDvEShCJhRm6JQQJ99CBACYeBjFXJ3w3AAAAACOGzRJb"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-12-01-preview"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-5-nano"

In [ ]:
# Import load_dotenv for environment variable loading
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
# Import libraries
import os
import json
import requests
import base64
from io import BytesIO
from openai import AzureOpenAI
from dotenv import load_dotenv
from IPython.display import HTML, display
load_dotenv()

In [ ]:
# Load environment variables and configure Azure OpenAI client
load_dotenv()
endpoint = os.getenv("ENDPOINT_URL") or os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment_name = os.getenv("DEPLOYMENT_NAME") or os.getenv("AZURE_OPENAI_DEPLOYMENT")
missing = [
    ("ENDPOINT_URL or AZURE_OPENAI_ENDPOINT", endpoint),
    ("AZURE_OPENAI_API_KEY", api_key),
    ("DEPLOYMENT_NAME or AZURE_OPENAI_DEPLOYMENT", deployment_name)
 ]
missing = [name for name, value in missing if not value]
if missing:
    missing_str = ", ".join(missing)
    raise RuntimeError("Missing required environment variables: " + missing_str + ", Check your .env file or Azure environment configuration.")
client = AzureOpenAI(azure_endpoint=endpoint, api_key=api_key, api_version="2025-01-01-preview")
print(f'✓ Azure OpenAI client configured with deployment: {deployment_name}')

In [ ]:
# Generate 10 SAT reading-writing questions and collect them
import time
import traceback
questions = []
for i in range(10):
    try:
        print(f"Generating SAT reading-writing question {i+1}/10...")
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[{
                'role': 'system',
                'content': 'You are an expert SAT reading-writing question writer. Create challenging reading comprehension, grammar, and writing questions that require careful analysis of passages, sentences, or context.'
            }, {
                'role': 'user',
                'content': '''Create a single SAT reading-writing question. It can be a reading comprehension, grammar, or writing question. Return ONLY a JSON object with this exact structure:
{
    "question": "the question text",
    "choices": ["A) choice 1", "B) choice 2", "C) choice 3", "D) choice 4"],
    "passage": "the passage or sentence to analyze (if applicable)"
}
IMPORTANT: In the passage, specify ONLY the text to analyze, NOT the answer or explanation.
Make the question challenging but solvable from the passage or context.'''
            }],
            response_format={'type': 'json_object'}
        )
        question_data = json.loads(response.choices[0].message.content)
        # Fix keys if needed
        if 'question' not in question_data:
            print("⚠️ WARNING: Response missing expected keys!")
            print(f"Keys received: {list(question_data.keys())}")
            if 'final' in question_data:
                if isinstance(question_data['final'], str):
                    question_data = json.loads(question_data['final'])
                else:
                    question_data = question_data['final']
            elif 'problem' in question_data:
                question_data['question'] = question_data['problem']
            if 'answer_choices' in question_data and 'choices' not in question_data:
                question_data['choices'] = question_data['answer_choices']
            if 'passage_text' in question_data and 'passage' not in question_data:
                question_data['passage'] = question_data['passage_text']
        print(f"Question: {question_data['question'][:100]}...")
        print(f"Passage: {question_data.get('passage', '')[:100]}...")
        questions.append(question_data)
    except Exception as e:
        print(f"Error generating question {i+1}: {e}")
        traceback.print_exc()
print(f"Total questions generated: {len(questions)}")


In [ ]:
# Debug cell: Print last error if any
import traceback
try:
    raise
except Exception as e:
    print('Error:', e)
    traceback.print_exc()
